# Olist Brazilian E-Commerce — Business Intelligence Analysis

## Project Overview
This notebook starts v0.1.0: **Data Cleaning & Data Integration**. The Olist dataset is relational, so we first inspect and validate each table before deciding how to join them.

**Execution environment:** GitHub Codespaces / VS Code. This notebook does not use Google Colab or Google Drive.


## 1. Setup Environment
Import the libraries needed for data manipulation and initial visualization.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Pandas version: {pd.__version__}')


## 2. Project Paths
Resolve the repository root so the notebook works whether the Codespace opens the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError(
        'Project root not found. Open this notebook inside the olist-bi-project repository.'
    )

ROOT = find_project_root()
RAW_DATA = ROOT / 'data' / 'raw'
PROCESSED_DATA = ROOT / 'data' / 'processed'

print(f'Project root : {ROOT}')
print(f'Raw data    : {RAW_DATA}')
print(f'Processed   : {PROCESSED_DATA}')


## 3. Load Datasets
The Olist dataset contains 9 CSV tables. The raw files must already be placed in `data/raw/` before running this section.


In [ ]:
file_names = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {}
missing_files = []

for name, filename in file_names.items():
    filepath = RAW_DATA / filename
    try:
        dfs[name] = pd.read_csv(filepath)
        print(f'Loaded {name:20s} | Shape: {dfs[name].shape}')
    except FileNotFoundError:
        missing_files.append(filename)
        print(f'File not found: {filename}')

print(f'\nTotal datasets loaded: {len(dfs)} / {len(file_names)}')
if missing_files:
    print(f'Missing files: {missing_files}')


## 4. Quick Overview per Table
Before cleaning anything, inspect row counts, columns, data types, sample records, and missing values. The findings here will determine the cleaning strategy for each table.


In [ ]:
for name, df in dfs.items():
    print('\n' + '=' * 70)
    print(f'DATASET: {name.upper()}')
    print('=' * 70)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'Columns: {list(df.columns)}')
    print('\nData types:')
    print(df.dtypes.value_counts().to_string())
    print('\nFirst 2 rows:')
    print(df.head(2))
    print('\nMissing values (%):')
    missing = (df.isna().mean() * 100).sort_values(ascending=False)
    print(missing[missing > 0].to_string() if (missing > 0).any() else 'No missing values')


## 5. Summary of All Tables


In [ ]:
table_info = pd.DataFrame({
    'Table': list(dfs.keys()),
    'Rows': [df.shape[0] for df in dfs.values()],
    'Columns': [df.shape[1] for df in dfs.values()]
})

table_info


## 6. Primary Key Uniqueness Check
Validate the identifiers that are expected to be unique before joining tables. A non-unique key can multiply rows during a merge.


In [ ]:
pk_checks = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id'
}

for table, pk in pk_checks.items():
    if table in dfs:
        df = dfs[table]
        n_rows = len(df)
        n_unique = df[pk].nunique()
        duplicates = n_rows - n_unique
        status = 'UNIQUE' if duplicates == 0 else f'{duplicates:,} duplicates'
        print(f'{table:15s} | {pk:20s} | {n_rows:7,} rows | {n_unique:7,} unique | {status}')


## 7. Cross-Table Relationship Validation
Start with a basic relationship check: every `order_id` in `order_items` should have a corresponding record in `orders`. Additional relationships will be validated after we understand the actual table grains.


In [ ]:
if {'orders', 'order_items'}.issubset(dfs):
    orders_ids = set(dfs['orders']['order_id'])
    items_ids = set(dfs['order_items']['order_id'])
    orphan_items = items_ids - orders_ids

    print(f'Orders unique IDs      : {len(orders_ids):,}')
    print(f'Order item unique IDs  : {len(items_ids):,}')
    print(f'Orphan order IDs       : {len(orphan_items):,}')
else:
    print('Load orders and order_items before running this validation.')


## 8. Next Steps
This notebook currently covers the **initial assessment** phase of v0.1.0.

After reviewing the actual results, we will continue with:
- handling missing values based on evidence
- correcting data types, especially date columns
- checking duplicate rows beyond primary-key uniqueness
- validating additional cross-table relationships
- deciding which cleaned / analytical tables should be saved to `data/processed/`

**No large multi-table merge will be performed until each table has been individually validated.**
